# Project 1: Digit Recogniser App ✍️ -> 🤖 -> 🌍

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anshid/MathMeetsML/blob/main/future-of-mathematicians/notebooks/01_digit_recogniser_app.ipynb)

> **From your syllabus**
> - *Linear Algebra*: a picture is a vector in $\mathbb R^{64}$, distance is a norm
> - *ML-I*, Unit III: logistic (softmax) regression
> - *Convex Optimization*: training = minimising a convex loss

**Your project:** in about 70 minutes you will go from raw data to a **live web app** that anyone with the link can use:
draw a digit with your mouse, your model reads it. Every section ends with a small win ✔.

⏱ about 75 minutes · 4 exercises. Short on time at minute 140? Protect Stage 5 (the app) and do Stage 6 at home.

| Stage | You build | Win |
|---|---|---|
| 1 | setup | the computer is ready |
| 2 | look at the data | a picture is a matrix |
| 3 | your first model, from scratch | beat a coin flip by a mile |
| 4 | a stronger model | ~97% on unseen digits |
| 5 | the app | a public link to your own AI |
| 6 | the project files | a GitHub-ready, CV-ready project |

<details open>
<summary><b>🧭 How this notebook works (30 seconds)</b></summary>

1. **Run a cell:** click it, press **Shift+Enter**. Go **top to bottom, in order**. Collapsed cells titled *"just run me"* must be run too (you do not need to read them).
2. **`____` means your turn.** Replace each `____` with your answer (the **Hint** above the cell helps). Do **not** use *Run all* before the blanks are filled.
3. **Red text?** Read only its **last line**, then check your blank and the hint. Still stuck: ask your squad, then raise a hand. Mistakes are the point.
4. **Core** is the goal for the session. **Stretch** is for when you finish early. **Quest** is take-home. Falling behind? Skip to the next ✔ cell: the maths box matters more than the code.
5. **Colab:** use *File ▸ Save a copy in Drive* to keep your work (the original is read-only).
</details>

> 📊 **Progress tracking: please read.** Run the next cell once: a box will ask for your **name** and **roll number** (press Enter after each). It then draws your progress map and records which exercises you finish. It sends your **name, roll number, exercise ID and time** to the instructor's progress sheet for this workshop, nothing else (no files, nothing from your Google account). Its code is collapsed to save space but is all there to read (Colab: the cell's **⋮** menu → *Show code*; JupyterLab: click the collapsed input). Prefer not to be tracked? Type `anonymous` as your name, or set `TRACKING = False`; the map still works for you. The same cell also turns the error message for an unfilled `____` into a friendly hint.
>
> Colab may ask *"Do you trust this code?"* because this notebook comes from GitHub (`anshid/MathMeetsML`, open source, MIT licence). That is expected: choose **Run anyway**.

In [ ]:
#@title Progress map (just run me)
# Records which exercises you finish so you can see your progress map and the instructor can see how the class is doing.
# Sent: your name, roll number, exercise id and time. Nothing else. Type "anonymous" as your name (or set TRACKING = False) to send nothing.
TRACKING = True
MISSION = 1

import json, os, time, base64, urllib.parse, urllib.request
from IPython.display import HTML, display, update_display

CLOUD_URL = "https://script.google.com/macros/s/AKfycbxekbOkIE5oRVA319mhEENS1phVLt7gPfRUernniP8kMopgIqUNHNN37xjF_3XZEuQWwQ/exec"
TOKEN = "3F5y-xKcPbZZPT49"
LAN_URL = os.environ.get("MML_LAN", "")
MISSIONS = {"1": {"title": "Digit App", "checkpoints": ["p01-ready", "p01-first-model", "p01-model", "p01-app", "p01-done"]}, "2": {"title": "Photo App", "checkpoints": ["p02-svd", "p02-ey", "p02-app", "p02-done"]}, "3": {"title": "Risk App", "checkpoints": ["p03-model", "p03-threshold", "p03-app", "p03-done"]}}
LABEL, UNIT = "P", "Project"
ORDER = sorted(MISSIONS, key=int)
STORE = "progress.json"
SINKS = [u for u in (CLOUD_URL, LAN_URL) if u]


def _load():
    try:
        with open(STORE) as f:
            return json.load(f)
    except Exception:
        return {"name": "", "roll": "", "done": {}, "queue": []}


def _save(s):
    try:
        with open(STORE, "w") as f:
            json.dump(s, f)
    except Exception:
        pass


def _who(s):
    if s["roll"]:
        return
    if os.environ.get("MML_TRACKING") == "off":
        s["name"], s["roll"] = "anonymous", "-"
        return
    try:
        s["name"] = input("Your name (or 'anonymous'): ").strip() or "anonymous"
        s["roll"] = input("Roll number: ").strip() or "-"
    except Exception:
        s["name"], s["roll"] = "anonymous", "-"


def _sending(s):
    return TRACKING and s["name"].lower() != "anonymous" and os.environ.get("MML_TRACKING") != "off"


def _post(url, ev):
    body = json.dumps(dict(ev, token=TOKEN)).encode()
    urllib.request.urlopen(urllib.request.Request(url, body, {"Content-Type": "text/plain"}), timeout=2).read()


def sync():
    """Send queued exercises to the instructor. Silent if offline; they stay queued."""
    s = _load()
    if _sending(s):
        dead = set()
        for ev in s["queue"]:
            for u in SINKS:
                if u in dead or u in ev["sent"]:
                    continue
                try:
                    _post(u, ev)
                    ev["sent"].append(u)
                except Exception:
                    dead.add(u)
        s["queue"] = [e for e in s["queue"] if any(u not in e["sent"] for u in SINKS)]
    _save(s)
    return s


def progress_code():
    """Offline? Paste this code to the instructor (chat or USB) instead."""
    s = _load()
    print(base64.b64encode(json.dumps({"name": s["name"], "roll": s["roll"], "done": s["done"]}).encode()).decode())


def _status(s, n):
    ids = MISSIONS[str(n)]["checkpoints"]
    got = sum(1 for c in ids if c in s["done"])
    if got == len(ids):
        return "done", got, len(ids)
    if n == MISSION or got:
        return "now", got, len(ids)
    return "todo", got, len(ids)


def _html(s):
    cols = {"done": "#2e9e5b", "now": "#3b82f6", "todo": "#8a8f98"}
    parts, stats = [], [_status(s, int(k)) for k in ORDER]
    for i, (st, got, tot) in enumerate(stats):
        n, x = int(ORDER[i]), 45 + i * 108
        title = MISSIONS[str(n)]["title"]
        if i:
            ok = stats[i - 1][0] == "done"
            parts.append(f'<line x1="{x - 108 + 20}" y1="40" x2="{x - 20}" y2="40" stroke="{cols["done"] if ok else cols["todo"]}" stroke-width="3" stroke-dasharray="{"0" if ok else "5 4"}"/>')
        if st == "done":
            parts.append(f'<circle cx="{x}" cy="40" r="18" fill="{cols[st]}"/><path d="M{x - 8} 40 l6 7 l11 -14" stroke="#fff" stroke-width="3.5" fill="none" stroke-linecap="round" stroke-linejoin="round"/>')
        elif st == "now":
            parts.append(f'<circle cx="{x}" cy="40" r="18" fill="none" stroke="{cols[st]}" stroke-width="4"><animate attributeName="r" values="18;22;18" dur="1.8s" repeatCount="indefinite"/></circle><text x="{x}" y="45" text-anchor="middle" font-size="14" font-weight="700" fill="{cols[st]}">{got}/{tot}</text>')
        else:
            parts.append(f'<circle cx="{x}" cy="40" r="18" fill="none" stroke="{cols[st]}" stroke-width="2.5" stroke-dasharray="4 3"/><text x="{x}" y="45" text-anchor="middle" font-size="14" fill="{cols[st]}">{n}</text>')
        parts.append(f'<text x="{x}" y="82" text-anchor="middle" font-size="12" fill="currentColor" opacity="{1 if st != "todo" else 0.55}">{LABEL}{n} {title}</text>')
    finished = [k for k, t in zip(ORDER, stats) if t[0] == "done"]
    todo = [f"{LABEL}{k}" for k, t in zip(ORDER, stats) if t[0] == "todo"]
    now = stats[ORDER.index(str(MISSION))]
    line = f"<b>{len(finished)} of {len(stats)} {UNIT.lower()}s finished</b> &middot; now in <b>{UNIT} {MISSION}</b> ({now[1]}/{now[2]} exercises)"
    line += f" &middot; not visited yet: {', '.join(todo)}" if todo else f" &middot; every {UNIT.lower()} visited"
    who = f"{s['name']} ({s['roll']})" if s["roll"] else ""
    return (f'<div style="font-family:sans-serif;max-width:700px"><svg viewBox="0 0 640 96" width="100%" role="img" aria-label="Progress map">{"".join(parts)}</svg>'
            f'<div style="font-size:14px">{line}</div><div style="font-size:12px;opacity:.6">{who}</div></div>')


_shown = []


def show_tracker():
    html = HTML(_html(_load()))
    if _shown:
        try:
            update_display(html, display_id="mml-tracker")
            return
        except Exception:
            pass
    display(html, display_id="mml-tracker")
    _shown.append(1)


def checkpoint(cid):
    """Call after an exercise passes: saves it, tells the instructor (if online), refreshes the map."""
    s = _load()
    _who(s)
    if cid not in s["done"]:
        s["done"][cid] = time.strftime("%H:%M:%S")
        s["queue"].append({"name": s["name"], "roll": s["roll"], "mission": MISSION, "checkpoint": cid, "ts": s["done"][cid], "sent": []})
    _save(s)
    sync()
    show_tracker()


def _start():
    s = _load()
    _who(s)
    if CLOUD_URL and _sending(s):  # Colab gives every notebook a fresh disk, so ask what you already finished elsewhere
        try:
            q = urllib.parse.urlencode({"roll": s["roll"], "token": TOKEN})
            for c in json.load(urllib.request.urlopen(CLOUD_URL + "?" + q, timeout=2)).get("done", []):
                s["done"].setdefault(c, "earlier")
        except Exception:
            pass
    _save(s)
    show_tracker()


def _friendly_blank(shell, etype, value, tb, tb_offset=None):
    """An unfilled blank (four underscores) raises a cryptic NameError/AttributeError. Say what it means; anything else is shown as usual."""
    blank = "_" * 4
    try:
        if blank in str(value):
            stb = ["✋ You have not filled in a blank yet.",
                   f"   Find the {blank} in this cell, replace it with your answer (the Hint above the cell helps), then run the cell again."]
        else:
            stb = shell.InteractiveTB.structured_traceback(etype, value, tb, tb_offset=tb_offset)
        shell._showtraceback(etype, value, stb)
    except Exception:
        print(f"{etype.__name__}: {value}")


try:
    get_ipython().set_custom_exc((NameError, AttributeError), _friendly_blank)
except Exception:
    pass  # plain Python or an unusual front end: the normal error message is fine


_start()

## Stage 1. Setup
Click the cell below and press **Shift+Enter**.

In [ ]:
#@title ▶ Setup (just run me; code hidden)
import os, sys, subprocess, time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

try:
    import gradio as gr
except ImportError:                                   # Colab: install the app library once (about 30 s)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gradio==6.31.0"], check=True)
    import gradio as gr

CHECK = os.environ.get("MML_CHECK") == "1"           # instructor's automatic test: build the app but do not launch it
np.random.seed(0)
print("numpy", np.__version__, "| gradio", gr.__version__)
print("Win #1: your lab computer is ready to build an AI app ✔")
checkpoint("p01-ready")

## Stage 2. Look at the data
1,797 handwritten digits, each an $8\times 8$ grid of numbers from 0 (white) to 16 (black). Flattened, each picture is a vector $x\in\mathbb R^{64}$.

In [ ]:
digits = load_digits()
X, y = digits.data, digits.target
print("pictures:", X.shape[0], "| numbers per picture:", X.shape[1])

fig, axes = plt.subplots(1, 10, figsize=(12, 1.6))
for d, ax in enumerate(axes):
    ax.imshow(digits.images[np.where(y == d)[0][0]], cmap="gray_r"); ax.set_title(d); ax.axis("off")
plt.show()

order = np.random.permutation(len(X))              # shuffle, then split
train, test = order[:1300], order[1300:]
print("train:", len(train), "| test (never seen while learning):", len(test))

**Baseline first.** A professional always asks: *what accuracy would a silly rule get?* Always guessing the most common digit gives about 10%. Anything we build must beat that.

In [ ]:
most_common = np.bincount(y[train]).argmax()
print("always guess", most_common, "-> accuracy", round(np.mean(y[test] == most_common), 3))

## Stage 3. Your first model, from scratch: nearest average
Idea: compute the **average picture** of each digit, $\mu_d=\frac1{n_d}\sum_{y_i=d}x_i$. A new picture gets the label of the closest average, $\hat y=\arg\min_d\|x-\mu_d\|_2$.

Fill in the blank: the mean of the training pictures whose label is `d`.

<details><summary>Hint</summary>

`X[train][y[train] == d]` selects the training pictures of digit `d`. `.mean(axis=0)` averages them row-wise.
</details>

In [ ]:
means = np.array([____ for d in range(10)])          # TODO: average training picture of digit d

fig, axes = plt.subplots(1, 10, figsize=(12, 1.6))
for d, ax in enumerate(axes):
    ax.imshow(means[d].reshape(8, 8), cmap="gray_r"); ax.set_title(f"avg {d}"); ax.axis("off")
plt.show()

Now classify every test picture by its nearest average. Fill in the distance from each picture to each mean.

<details><summary>Hint</summary>

The array has shape (test pictures, 10 digits, 64 pixels), so the 64 pixels are the last axis. Axes are counted from 0.
</details>

In [ ]:
dists = np.linalg.norm(X[test][:, None, :] - means[None, :, :], axis=____)   # TODO: which axis holds the 64 pixels?
pred = dists.argmin(axis=1)
acc_first = np.mean(pred == y[test])
print("nearest-average accuracy:", round(acc_first, 3))
assert acc_first > 0.85, "check the axis: shape is (test pictures, 10 digits, 64 pixels)"
print("Win #2: a model you wrote yourself reads handwriting ✔")
checkpoint("p01-first-model")

## Stage 4. A stronger model: softmax regression
Each digit $d$ gets a weight vector $w_d$. The model turns scores into probabilities,
$$P(y=d\mid x)=\frac{e^{w_d^\top x+b_d}}{\sum_k e^{w_k^\top x+b_k}},$$
and training minimises the **cross-entropy** loss, which is **convex** in $(w,b)$ (Convex Optimization). scikit-learn does the minimising; you choose and check.

Fill in the training call.

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=2000)
model.____(X[train], y[train])                     # TODO: the method that trains (learns w and b)
acc = model.score(X[test], y[test])
print("softmax regression accuracy:", round(acc, 3))
assert acc > 0.93
print("Win #3: about 97% on digits the model has never seen ✔")
checkpoint("p01-model")

**Where does it go wrong?** A confusion matrix: row = true digit, column = predicted digit. Off-diagonal = mistakes.

In [ ]:
#@title ▶ Confusion matrix and weight pictures (just run me; code hidden)
from sklearn.metrics import ConfusionMatrixDisplay
ConfusionMatrixDisplay.from_estimator(model, X[test], y[test], cmap="Blues", colorbar=False)
plt.title("which digits get confused?"); plt.show()

fig, axes = plt.subplots(1, 10, figsize=(12, 1.6))   # the learned weights w_d, seen as pictures
for d, ax in enumerate(axes):
    ax.imshow(model.coef_[d].reshape(8, 8), cmap="coolwarm"); ax.set_title(f"w_{d}"); ax.axis("off")
plt.show()

## Stage 5. The app
A mouse drawing is a large image, but the model eats 64 numbers. `to_8x8` crops your digit, centres it, shrinks it to $8\times8$ (averaging blocks of pixels) and rescales to 0..16. Run the cell below (its code is collapsed: **⋮ → Show code** if you are curious).

In [ ]:
#@title ▶ Image preprocessing helper (just run me; code hidden)
from PIL import Image

def to_8x8(drawing):
    """Turn a big drawing (RGB or RGBA numpy array) into the 64 numbers the model expects."""
    a = np.asarray(drawing, dtype=float)
    if a.ndim == 2:
        a = np.stack([a] * 3, axis=-1)
    ink = 1 - a[..., :3].mean(axis=2) / 255            # dark pixels = ink
    if a.shape[2] == 4:
        ink = ink * a[..., 3] / 255                     # transparent pixels = no ink
    rows, cols = np.where(ink > 0.2)
    if len(rows) == 0:
        return np.zeros(64)
    ink = ink[rows.min():rows.max() + 1, cols.min():cols.max() + 1]   # crop to the digit
    h, w = ink.shape
    side = max(h, w)                                    # centre it in a square, like the training digits
    square = np.zeros((side, side))
    square[(side - h) // 2:(side - h) // 2 + h, (side - w) // 2:(side - w) // 2 + w] = ink
    small = np.asarray(Image.fromarray((square * 255).astype(np.uint8)).resize((8, 8), Image.BOX), dtype=float)
    return (small / small.max() * 16).ravel()           # same 0..16 scale as the dataset

**Test before you ship.** We blow up real test digits into big fake drawings, push them through `to_8x8`, and check the model still reads them.

In [ ]:
#@title ▶ Test helper: fake drawings (just run me; code hidden)
def fake_drawing(img8):
    big = np.pad(np.kron(img8, np.ones((35, 35))), 60)            # 8x8 -> 400x400 with a white border
    return np.stack([255 - big * 15.9] * 3, axis=-1).astype(np.uint8)

ok = np.mean([model.predict([to_8x8(fake_drawing(digits.images[i]))])[0] == y[i] for i in test[:200]])
print("accuracy on big fake drawings:", round(ok, 3))
assert ok > 0.85

Now connect the pieces. `predict` receives the sketch, and must return a probability for each digit. Fill in the blank: the model's probabilities for one picture.

<details><summary>Hint</summary>

`model.predict_proba([features])` returns a 1 x 10 array. Take row `[0]`.
</details>

In [ ]:
def predict(sketch):
    if sketch is None or sketch.get("composite") is None:
        return {}
    features = to_8x8(sketch["composite"])
    probs = ____                                   # TODO: probabilities for the 10 digits, a length-10 array
    return {str(d): float(p) for d, p in enumerate(probs)}

demo = gr.Interface(fn=predict, inputs=gr.Sketchpad(type="numpy", label="Click the brush icon, then draw one digit, big and centred",
                                      canvas_size=(400, 400), layers=False, transforms=(),
                                      brush=gr.Brush(default_size=28, colors=["#000000"], color_mode="fixed")),
                    outputs=gr.Label(num_top_classes=3, label="The model thinks..."),
                    title="Digit Recogniser", description="Softmax regression on 8x8 digits. Built at the MathMeetsML workshop.",
                    flagging_mode="never")

guess = predict({"composite": fake_drawing(digits.images[test[0]])})   # test the app function before launching
assert max(guess, key=guess.get) == str(y[test[0]]), guess
if CHECK:
    print("check mode: app built, not launched")
else:
    demo.launch(share=True)                          # wait ~10 s, then open the public link that appears
print("Win #4: your AI is live on the web. Send the link to a friend ✔")
checkpoint("p01-app")

**In the app:** click the ✏️ **brush** icon on the left of the canvas first, draw, then press **Submit**. **Clear** to try again.

> **Honest limits (say this in your showcase).** The model learnt from 8x8 scans of digits written by ~40 people in the 1990s. Try a **1 with a flag at the top**: it is often read as a 3, because the training 1s are mostly plain thick bars. That is **distribution shift**, the most common reason real AI products fail. Draw big, thick and centred.

The `share=True` link lives for about 72 hours. For a **permanent** link, Stage 6 packs your project for GitHub and Hugging Face Spaces.

## Stage 6. Make it a real project
A CV-ready project is a **repository** with three files: the app, its requirements and a README that explains the maths. Edit `YOUR_NAME` in the first cell, then run both cells.

In [ ]:
YOUR_NAME = "Your Name"                            # TODO (optional): your name for the README

In [ ]:
#@title ▶ Write the project files (just run me; code hidden)
APP_PY = r'''# app.py - Digit Recogniser (MathMeetsML project). Run locally: python app.py
import numpy as np
import gradio as gr
from PIL import Image
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression

digits = load_digits()
model = LogisticRegression(max_iter=2000).fit(digits.data, digits.target)   # trains in about a second


def to_8x8(drawing):
    a = np.asarray(drawing, dtype=float)
    if a.ndim == 2:
        a = np.stack([a] * 3, axis=-1)
    ink = 1 - a[..., :3].mean(axis=2) / 255
    if a.shape[2] == 4:
        ink = ink * a[..., 3] / 255
    rows, cols = np.where(ink > 0.2)
    if len(rows) == 0:
        return np.zeros(64)
    ink = ink[rows.min():rows.max() + 1, cols.min():cols.max() + 1]
    h, w = ink.shape
    side = max(h, w)
    square = np.zeros((side, side))
    square[(side - h) // 2:(side - h) // 2 + h, (side - w) // 2:(side - w) // 2 + w] = ink
    small = np.asarray(Image.fromarray((square * 255).astype(np.uint8)).resize((8, 8), Image.BOX), dtype=float)
    return (small / small.max() * 16).ravel()


def predict(sketch):
    if sketch is None or sketch.get("composite") is None:
        return {}
    probs = model.predict_proba([to_8x8(sketch["composite"])])[0]
    return {str(d): float(p) for d, p in enumerate(probs)}


demo = gr.Interface(fn=predict, inputs=gr.Sketchpad(type="numpy", label="Click the brush icon, then draw one digit, big and centred",
                                      canvas_size=(400, 400), layers=False, transforms=(),
                                      brush=gr.Brush(default_size=28, colors=["#000000"], color_mode="fixed")),
                    outputs=gr.Label(num_top_classes=3, label="The model thinks..."),
                    title="Digit Recogniser", description="Softmax regression on 8x8 digits. Built at the MathMeetsML workshop.",
                    flagging_mode="never")

if __name__ == "__main__":
    demo.launch()'''

README = f'''---
title: Digit Recogniser
emoji: ✍️
sdk: gradio
sdk_version: {gr.__version__}
app_file: app.py
license: mit
---
# Digit Recogniser (softmax regression from first principles)

Draw a digit; a softmax-regression model trained on the UCI 8x8 digits dataset reads it.

**Maths:** each image is a vector $x\\in\\mathbb R^{{64}}$; $P(y=d\\mid x)\\propto e^{{w_d^\\top x+b_d}}$; training minimises the convex
cross-entropy loss. Baseline (most common class) about 10%, nearest-average classifier {acc_first:.1%}, softmax regression {acc:.1%} test accuracy.

**Limits:** trained on 1990s 8x8 scans; thin or off-centre drawings are out of distribution.

Built by {YOUR_NAME} at the MathMeetsML workshop. Code MIT.
'''
REQS = "scikit-learn==1.9.1\nnumpy\npillow\n"

os.makedirs("digit-recogniser", exist_ok=True)
for fname, text in {"app.py": APP_PY, "README.md": README, "requirements.txt": REQS}.items():
    with open(os.path.join("digit-recogniser", fname), "w", encoding="utf-8") as f:
        f.write(text.strip() + "\n")
print(sorted(os.listdir("digit-recogniser")))

**Test the packaged app** (the same code, now in one file), then download the folder as a zip.

In [ ]:
#@title ▶ Test and zip the project (just run me; code hidden)
import runpy, shutil
packed = runpy.run_path("digit-recogniser/app.py", run_name="packaged")       # runs app.py without launching it
guess = packed["predict"]({"composite": fake_drawing(digits.images[test[1]])})
assert max(guess, key=guess.get) == str(y[test[1]]), guess
shutil.make_archive("digit-recogniser", "zip", "digit-recogniser")
print("packaged app works ✔  ->  digit-recogniser.zip")
if not CHECK:
    try:
        from google.colab import files
        files.download("digit-recogniser.zip")
    except ImportError:
        print("Not on Colab: the zip is next to this notebook.")
print("Win #5: a complete, deployable project ✔")
checkpoint("p01-done")

## Deploy it permanently (10 minutes, after the session)
Follow `future-of-mathematicians/deploy-guide.md`: unzip -> new GitHub repository -> upload the 3 files -> new Hugging Face Space (Gradio) -> upload the same files. Your app gets a permanent URL to put on your CV.

## Stretch
- Add a second output: show the $8\times8$ picture the model actually sees (`gr.Image`). Does it look like the training digits?
- Swap `LogisticRegression` for `sklearn.neighbors.KNeighborsClassifier(3)`. Which wins on your own drawings?

## You just used...
- vectors and **norms** (nearest-average classifier)
- **softmax regression** and a **convex** loss (ML-I, Convex Optimization)
- **train/test splits, baselines, confusion matrices**: how professionals evaluate a model
- **deployment**: turning maths into a product someone else can use

## Your CV lines
> *Built and deployed a handwritten-digit recogniser web app (Python, scikit-learn, Gradio): nearest-centroid baseline from scratch, softmax regression at ~97% test accuracy, hosted on Hugging Face Spaces.*

Take-home projects: **Project 2** (photo compressor app, SVD) and **Project 3** (risk estimator app, probability).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anshid/MathMeetsML/blob/main/future-of-mathematicians/notebooks/02_photo_compressor_app.ipynb)